# 2. Run and inspect a quick experiment

This notebook calls the package CLI instead of embedding a second training loop. The expensive cells are opt-in so opening the notebook cannot accidentally start a GPU job. Quick results are tutorial-only.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

from vision_bench.config import load_project_config
from vision_bench.engine import run_directory

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
PRESET = "quick"
DEVICE = "cuda"  # change to 'mps' or 'cpu' for a local learning run
RUN_TRAINING = False  # read the next cell, then deliberately change to True

## Train one ViT run

The command first prepares CIFAR-100, then trains or resumes the configured ViT run. Leave `RUN_TRAINING` false when executing notebooks in CI or on a machine without an appropriate accelerator.

In [ ]:
if RUN_TRAINING:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "vision_bench",
            "prepare-data",
            "--preset",
            PRESET,
            "--project-root",
            str(PROJECT_ROOT),
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "vision_bench",
            "train",
            "--preset",
            PRESET,
            "--project-root",
            str(PROJECT_ROOT),
            "--model",
            "vit",
            "--mode",
            "standard",
            "--seed",
            "42",
            "--device",
            DEVICE,
        ],
        check=True,
    )
else:
    print("Training is disabled. Set RUN_TRAINING = True after choosing a suitable device.")

## Inspect per-epoch behavior

This cell reads an existing artifact if one is present. Each JSON Lines record is independently parseable.

In [ ]:
project = load_project_config(PRESET, PROJECT_ROOT)
vit_run = next(run for run in project.runs if run.model == "vit")
metrics_path = run_directory(ARTIFACT_ROOT, project, vit_run) / "metrics.jsonl"
if metrics_path.exists():
    records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line]
    learning_curve = pd.DataFrame(
        [
            {
                "epoch": row["epoch"],
                "train_loss": row["train"]["loss"],
                "train_top1": row["train"]["top1"],
                "validation_top1": row["validation"]["top1"],
                "minutes": row["cumulative_seconds"] / 60,
            }
            for row in records
        ]
    )
    display(learning_curve)
else:
    print(f"No metrics yet: {metrics_path}")

## Complete the quick suite

After the one-run walkthrough, the terminal command `uv run vision-bench suite --preset quick --device cuda` trains the teacher first and then every configured candidate. Re-running it resumes incomplete runs and skips complete ones.